# C1 Session 1 — Learning from Data and the Types of Learning Tasks

*Machine Learning Fundamentals, session 1 of 3 (~75 min). Prerequisite:
F1-scientific-python.*

This session establishes what "learning from data" means, then builds the
first and most consequential distinction in the field: tasks where the data
carries answers (supervised) versus tasks where it does not (unsupervised),
with clustering as the flagship unlabeled task. Each section ends with a
checkpoint; answers are collected at the end of this notebook.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

## 1. Learning from Data

**Motivation.** Suppose you run a fruit-packing station and need a machine to
separate mandarins from oranges. One approach: a person studies the fruit and
writes a rule by hand — *"if it weighs 140 grams or more, call it an orange."*
That works until the harvest changes, the fruit gets bigger or smaller, and a
human has to study the problem all over again and rewrite the rule.

**The machine-learning idea.** Instead of writing the rule ourselves, we show
the computer *examples with known answers* and let it work out the rule. When
conditions change, we do not rewrite anything — we hand over fresh examples
and the rule is recomputed.

**Worked example.** Here are six fruits whose species we already know
(label `0` = mandarin, `1` = orange). The computer "learns" a cutoff weight
by taking the midpoint between the two class averages:

In [ ]:
weights = np.array([112.0, 130.0, 118.0, 155.0, 162.0, 148.0])  # grams
labels  = np.array([0, 0, 0, 1, 1, 1])  # 0 = mandarin, 1 = orange

mean_mandarin = weights[labels == 0].mean()
mean_orange   = weights[labels == 1].mean()
cutoff = (mean_mandarin + mean_orange) / 2

print("mandarin average:", mean_mandarin)
print("orange average:  ", mean_orange)
print("learned cutoff:  ", cutoff)
print("verdict for a 143 g fruit:", "orange" if 143.0 >= cutoff else "mandarin")

Nobody typed the cutoff in — it came *out of the data*. If next season's
mandarins grow larger, we feed in new examples and the cutoff moves by
itself. That is the whole subject in miniature: **a rule learned from
examples, instead of a rule written by hand.**

### Checkpoint 1

1. Give one task where a hand-written rule is clearly the right tool, and one
   task where learning from examples is the better strategy. One sentence of
   justification each.
2. Suppose the orange examples had weighed 175, 182, and 168 grams instead.
   Without running code: does the learned cutoff move up or down, and why?

## 2. Supervised Learning: Data with Answers Attached

**Motivation.** The fruit example was easy to learn from for one reason: every
weight arrived *with its species attached*. The first question to ask about
any learning problem is: **do the examples come with answers or not?**

**Definition.** **Supervised learning** is learning from *labeled* examples.
Each example is a pair: measured inputs (the *features*) and the known answer
(the *label*). The goal is a rule that produces the right label for **new**
inputs it has never seen. The fruit cutoff was supervised learning: features
= weight, labels = species.

**Worked example.** Hours studied (feature) with a pass/fail label per
student:

In [ ]:
hours  = np.array([1.0, 2.0, 2.5, 4.0, 5.5, 6.0, 7.5, 8.0])  # features
passed = np.array([0,   0,   0,   0,   1,   1,   1,   1])    # labels (the answers)

for h, p in zip(hours, passed):
    print(f"studied {h:>4} h -> {'passed' if p else 'failed'}")

**A second worked example, with several features.** Features need not be a
single number. A clinic predicting whether a treatment will help might record
three measurements per patient — the feature array becomes 2-D, one row per
example, and the label array stays 1-D, one answer per row:

In [ ]:
# rows = patients; columns = (age, resting heart rate, hours of sleep)
features = np.array([[34.0, 62.0, 7.5],
                     [61.0, 81.0, 5.0],
                     [45.0, 70.0, 6.5],
                     [29.0, 58.0, 8.0]])
helped = np.array([1, 0, 1, 1])          # one label per row

print("features shape:", features.shape, " labels shape:", helped.shape)
print("patient 1's features:", features[1], "-> label", helped[1])

The shape contract — features `(n, d)`, labels `(n,)` — recurs through the
whole course. Whatever the task, "supervised" always means: **the answer we
want to predict is present in the data as a label column.**

### Checkpoint 2

1. A vet clinic records `(weight, temperature, age)` for each animal plus
   whether it recovered, and wants to predict recovery for new patients.
   Identify the features and the label. Is this supervised?
2. Predict a house's sale price from size and location, using last year's
   sales — each with its actual price. Supervised or unsupervised, and what
   single fact about the data decides it?

## 3. Unsupervised Learning and Clustering

**Motivation.** Many datasets are not so generous: measurements pile up with
no answer column at all. Nobody has tagged 50,000 songs by style; no one has
labeled which of a million customers belong to which "type" — the types have
not even been invented yet.

**Definition.** **Unsupervised learning** is learning from *unlabeled*
examples. There is no answer column. The goal is to find structure that is
already sitting in the data — groups, patterns, unusual points.

**Clustering** is the flagship unsupervised task: split the data into groups
(*clusters*) so that items in the same group are similar to each other.
Nobody tells the computer what the groups are, how they should be named, or
which item belongs where — it finds the grouping on its own.

**Worked example: clustering without labels.** The heights (in cm) of people
at a family picnic — no labels. Are there natural groups? Sort the values and
look for the biggest jump between neighbours — everything below the jump is
one group, everything above is another. `np.sort`, `np.diff`, and `np.argmax`
are all we need (as covered in F1-scientific-python):

In [ ]:
heights = np.array([131.0, 174.0, 128.0, 180.0, 136.0, 169.0, 125.0, 177.0, 133.0])

s = np.sort(heights)
gaps = np.diff(s)                # distance between each sorted neighbour pair
split = np.argmax(gaps)          # index of the biggest jump

group_a = s[:split + 1]
group_b = s[split + 1:]
print("sorted heights:", s)
print("gaps:          ", gaps)
print("group A:", group_a)
print("group B:", group_b)

The computer separated the picnic into two clusters — and it was never told
that "children" and "adults" exist. That is exactly what clustering
delivers: **the groups themselves**. What clustering can *not* deliver is the
meaning of the groups; a human looks at group A and group B and recognizes
them as kids and grown-ups. No labels in, groupings out, interpretation still
up to us.

### Checkpoint 3

1. Take 50,000 untagged songs and group together ones that sound alike —
   supervised or unsupervised, and which specific task is it?
2. In your head, apply the biggest-gap idea to `[2, 3, 9, 10, 11]`. Which
   two groups come out, and which gap decided it?

## 4. Recognizing Task Types

**Motivation.** Exam items and real projects rarely announce "this one is
supervised" — they describe a situation and expect you to classify it. A
quick test settles nearly every case:

| Question | If yes… |
|---|---|
| Do the training examples include the answer we want to produce? | supervised |
| Are we hunting for structure in answer-free data? | unsupervised |
| Specifically: are we splitting the data into groups of similar items? | clustering |

**Worked mini-cases.** Four descriptions, classified by the test:

1. *Predict tomorrow's peak electricity demand from past days' records, each
   including that day's actual peak.* The answer (actual peak) is in the data
   → **supervised**.
2. *Group 10,000 news articles by topic; no topic tags exist.* No answer
   column, hunting for groups → **unsupervised (clustering)**.
3. *Given photos each tagged "cat" or "dog", learn to tag new photos.* Tags
   are labels → **supervised**.
4. *From purchase histories alone, find groups of customers with similar
   habits.* No labels, groups wanted → **unsupervised (clustering)**.

Notice what does **not** matter: whether the data is big or small, numeric or
text, easy or hard. Only the presence of the answer in the data decides.

### Checkpoint 4

1. Learn to flag emails as spam using a folder of emails users already marked
   spam or not-spam. Classify the task.
2. A store has purchase histories and nothing else, and wants "customer types
   nobody has defined yet". Classify the task, and state exactly what
   deliverable the store will get from it.

## 5. Common Pitfalls (Task-Type Edition)

**Pitfall 1: "the algorithm found the children."** Clustering outputs
groups, never names. Broken claim: *"the clustering reported that group A is
the children."* The algorithm reported only membership; the interpretation
was added by a person. The fix is to keep the two contributions separate —
and if you want names attached, a human supplies them explicitly:

In [ ]:
# What the algorithm produced (from Section 3):
print("group A:", group_a)      # membership only -- no meaning attached
print("group B:", group_b)

# FIX: interpretation is a human step, recorded as such:
names = {"A": "children (interpreted by us)", "B": "adults (interpreted by us)"}
print(names)

**Pitfall 2: "we know the answers, so it's labeled."** A botanist may be able
to identify every flower in an untagged photo archive on sight — but until
those identifications are *recorded in the data as a label column*, the
dataset is unlabeled and supervised learning cannot start. Broken:
"the fraud team recognizes fraud when they see it, so our transaction log is
labeled." Fix: have the experts label examples first; *then* the task becomes
supervised. Supervision is a property of the dataset, not of anyone's head.

**Pitfall 3: "no labels means nothing can be done."** Unlabeled data is not
useless — Section 3 pulled real structure out of nine unlabeled numbers.
Broken: discarding a huge unlabeled dataset because "we can't train on it."
Fix: unsupervised tools can group it, surface unusual points, and even guide
*which* examples are worth paying humans to label.

### Checkpoint 5

1. A teammate says, "the clustering algorithm determined that cluster 2 is
   students who will drop out." Which part of that sentence did the algorithm
   actually produce, and which part did it not?
2. Your boss says, "our support staff can recognize angry emails instantly,
   so the email archive is labeled data." What concrete step is missing
   before a supervised angry-email flagger can be trained?

## 6. Exam Connections and Going Deeper

On the real Round 1 exam, this session's material appears in the "ML
concepts" cluster (see `reference/analysis.md`, topic-distribution table):
concept multiple-choice items — exactly five options A–E, sometimes with a
"cannot be determined" style distractor — that test precisely the
distinctions drilled above, at paraphrase level. The reliable strategy is the
Section 4 test: check where the answers live.

**Worked exam-style example.** *Reasoning is required. No coding is needed.*

Which ONE of the following tasks is **unsupervised**?

- **A.** Predict tomorrow's peak electricity demand from past days' records
  that include each day's actual peak.
- **B.** Learn to tag photos "cat" or "dog" from a collection of tagged
  photos.
- **C.** Flag emails as spam using folders that users already marked.
- **D.** Group 50,000 untagged product photos by visual similarity.
- **E.** Predict an apartment's rent from listings that include the rent.

*Worked solution.* Apply the answers-in-data test to each option. A: the
actual peak is the answer, and it is in the records — supervised. B: the tags
are labels — supervised. C: the user-marked folders are labels — supervised.
E: the rent is the answer and appears in the listings — supervised. D is the
only task with no answer column ("untagged"), and it asks for groups of
similar items — unsupervised (clustering). Answer: **D**. One test, applied
five times, settles the item — that is typical of the concept-MC register.

**Going deeper (optional).** The workflow this unit builds by hand — data,
task type, split, measure — is carried out with a standard toolkit on
realistic tabular datasets in **C4-classical-ml-practice**, where clustering
and supervised classifiers appear as ready-made components. Nothing there is
needed for this unit's practice set.

### Checkpoint 6

1. Write your own five-option A–E exam item that asks the reader to classify
   one described task, with exactly one correct option. (Then check: does
   every wrong option fail the answers-in-data test for a stated reason?)
2. In the worked example, one single question separated all five options.
   State it in your own words.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Hand-written rule: e.g. "a password must have at least 8 characters" — the
   condition is exact, known, and never drifts. Learning from examples: e.g.
   recognizing handwritten digits — nobody can write down exact conditions
   for "this is a 7", but labeled examples are easy to collect.
2. The cutoff moves **up**: the orange average rises to 175 g, so the
   midpoint between the class averages rises too.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Features: `(weight, temperature, age)`; label: whether the animal
   recovered. Yes — supervised: the answer is recorded with each example.
2. **Supervised** — each past sale carries the answer (its actual price).
   The deciding fact: the answer column exists in the data.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. **Unsupervised** — specifically clustering: no tags, and the goal is
   groups of similar songs.
2. Sorted: 2, 3, 9, 10, 11. Gaps: 1, 6, 1, 1. The gap 3→9 is largest, so the
   groups are `{2, 3}` and `{9, 10, 11}`.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. **Supervised** — the user-marked folder supplies the labels.
2. **Unsupervised (clustering).** The deliverable is the grouping itself:
   which customers are similar to which. What the types *mean*, and their
   names, the store's own analysts must supply.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. The algorithm produced cluster 2's *membership* — which students resemble
   one another. "Will drop out" is an interpretation (and a prediction) the
   algorithm never made; no labels about dropping out were ever involved.
2. The staff's recognitions must be recorded as labels: someone must go
   through a set of archived emails and mark each one angry / not-angry.
   Only then does a labeled dataset exist to train on.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. (Own work.) A sound item names one task per option, exactly one lacking an
   answer column (or exactly one having it), and each distractor is wrong for
   a reason you can state via the answers-in-data test.
2. "Does the training data already contain the answer the task wants to
   produce?" — yes → supervised; no → unsupervised.

</details>